# Analyse pretrained token embeddings

Train a model from the repository root using:

```sh
python runnables/run_pretraining.py
```

The launcher preserves the previous four-worker, 300-epoch defaults. Use `--processes 1` for a single worker and pass training options such as `--num-epochs 50` to override defaults. This notebook only loads checkpoints and visualises embeddings. Install dependencies from `requirements.txt` before opening it.


# UMAP Projection of Token Embeddings

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "src" / "model.py").is_file() and (path / "runnables").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the repository or one of its subdirectories.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.project_paths import SHARED_DATA_DIR, COHORT_DATA_DIR

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

try:
    import umap.umap_ as umap
except ImportError as exc:
    raise ImportError("Install 'umap-learn' in this kernel before running these cells.") from exc

from src.model import Transformer
from src.data_prep.build_sequences import load_vocab
from pathlib import Path
import re

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"

VOCAB_PATH = CHECKPOINT_DIR / 'vocab.json'
CHECKPOINT_PATH = CHECKPOINT_DIR / 'best.pt'

if not VOCAB_PATH.exists():
    raise FileNotFoundError(f'Missing vocab: {VOCAB_PATH}')
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f'Missing checkpoint: {CHECKPOINT_PATH}')

vocab = load_vocab(str(VOCAB_PATH))
state_dict = torch.load(CHECKPOINT_PATH, map_location='cpu')

d_model = int(state_dict['tok_emb.weight'].shape[1])
vocab_size = int(state_dict['tok_emb.weight'].shape[0])
layer_ids = {
    int(key.split('.')[2])
    for key in state_dict
    if key.startswith('transformer.layers.') and key.split('.')[2].isdigit()
}
if not layer_ids:
    raise ValueError('Could not infer the number of transformer layers from the checkpoint.')
ff_dim = int(state_dict['transformer.layers.0.linear1.weight'].shape[0])
if ff_dim % d_model != 0:
    raise ValueError('Could not infer ff_mult from the checkpoint.')

model = Transformer(
    vocab_size=vocab_size,
    d_model=d_model,
    nhead=4,
    num_layers=max(layer_ids) + 1,
    ff_mult=ff_dim // d_model,
    dropout=0.1,
    pad_idx=vocab.pad_idx,
)
model.load_state_dict(state_dict, strict=False)
model.eval()

ICD_PATTERN = re.compile(r'^[A-Z]\d\d$')
ATC_PATTERN = re.compile(r'^[A-Z]\d\d[A-Z]$')
LEADING_CODE_PATTERN = re.compile(r'^([A-Z]\d\d(?:[A-Z])?)\b')
SPECIAL_TOKENS = {'<pad>', '<unk>', '<male>', '<female>'}
AGE_GRID = [0.0, 20.0, 40.0, 60.0, 80.0]


def extract_token_code(token: str) -> str | None:
    token = str(token)
    if token in SPECIAL_TOKENS:
        return None
    match = LEADING_CODE_PATTERN.match(token)
    return match.group(1) if match else None


def token_family(token: str) -> str:
    code = extract_token_code(token)
    if code is None:
        token = str(token)
        if token in SPECIAL_TOKENS:
            return 'SPECIAL'
        return 'OTHER'
    if ICD_PATTERN.fullmatch(code):
        return 'ICD'
    if ATC_PATTERN.fullmatch(code):
        return 'ATC'
    return 'OTHER'


def token_chapter(token: str) -> str | None:
    code = extract_token_code(token)
    if code and code[0].isalpha():
        return code[0]
    token = str(token)
    if token and token[0].isalpha() and token not in SPECIAL_TOKENS:
        return token[0]
    return None


token_meta = pd.DataFrame({
    'token_id': np.arange(len(vocab.itos), dtype=np.int64),
    'token': vocab.itos,
})
token_meta['code'] = token_meta['token'].map(extract_token_code)
token_meta['family'] = token_meta['token'].map(token_family)
token_meta['chapter'] = token_meta['token'].map(token_chapter)
token_meta = token_meta[token_meta['family'].isin(['ICD', 'ATC'])].copy()
if token_meta.empty:
    raise ValueError(
        'No ICD/ATC tokens were found in the loaded vocabulary. Check the checkpoint vocab format or update the token parsing rules.'
    )
token_meta['chapter_label'] = token_meta['family'] + '-' + token_meta['chapter'].fillna('?')

token_embeddings = model.tok_emb.weight.detach().cpu().numpy()

age_tensor = torch.tensor(AGE_GRID, dtype=torch.float32).unsqueeze(1)
with torch.no_grad():
    age_vectors = model._age_encoding(age_tensor).squeeze(1).cpu().numpy()

print(f'Loaded checkpoint: {CHECKPOINT_PATH}')
print(f'Vocabulary size: {len(vocab)}')
print(token_meta['family'].value_counts())


In [ ]:
sns.set_theme(style='whitegrid')

UMAP_RANDOM_STATE = 0


def project_embeddings(embedding_matrix: np.ndarray, meta_df: pd.DataFrame, random_state: int | None = None) -> pd.DataFrame:
    if meta_df.empty:
        raise ValueError('project_embeddings received no tokens to project.')
    if random_state is None:
        random_state = UMAP_RANDOM_STATE
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=30,
        min_dist=0.1,
        metric='cosine',
        random_state=random_state,
    )
    coords = reducer.fit_transform(embedding_matrix[meta_df['token_id'].to_numpy()])
    out = meta_df.copy()
    out['umap_1'] = coords[:, 0]
    out['umap_2'] = coords[:, 1]
    return out


def make_palette(labels: list[str]) -> dict[str, tuple[float, float, float]]:
    palette = sns.color_palette('tab20', n_colors=max(len(labels), 1))
    return {label: palette[idx] for idx, label in enumerate(labels)}


def plot_projection(ax, df: pd.DataFrame, title: str, palette: dict[str, tuple[float, float, float]]) -> None:
    markers = {'ICD': 'o', 'ATC': 'X'}
    for family, marker in markers.items():
        fam_df = df[df['family'].eq(family)]
        if fam_df.empty:
            continue
        sns.scatterplot(
            data=fam_df,
            x='umap_1',
            y='umap_2',
            hue='chapter_label',
            palette=palette,
            marker=marker,
            s=34,
            alpha=0.8,
            linewidth=0,
            legend=False,
            ax=ax,
        )
    ax.set_title(title)
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')


def add_family_specific_subplot(ax, df: pd.DataFrame, family: str, title: str) -> None:
    fam_df = df[df['family'].eq(family)].copy()
    if fam_df.empty:
        ax.set_axis_off()
        return
    chapter_labels = sorted(fam_df['chapter_label'].dropna().unique().tolist())
    fam_palette = make_palette(chapter_labels)
    sns.scatterplot(
        data=fam_df,
        x='umap_1',
        y='umap_2',
        hue='chapter_label',
        palette=fam_palette,
        s=36,
        alpha=0.85,
        linewidth=0,
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    ax.legend(title='Chapter', bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)


combined_labels = sorted(token_meta['chapter_label'].dropna().unique().tolist())
combined_palette = make_palette(combined_labels)

token_only_umap = project_embeddings(token_embeddings, token_meta, random_state=UMAP_RANDOM_STATE)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 7), constrained_layout=True)
plot_projection(axes[0], token_only_umap, 'Token Embeddings Only: ICD and ATC', combined_palette)
add_family_specific_subplot(axes[1], token_only_umap, 'ICD', 'Token Embeddings Only: ICD Chapters')
add_family_specific_subplot(axes[2], token_only_umap, 'ATC', 'Token Embeddings Only: ATC Chapters')
plt.show()

age_projection_frames = []
for age_value, age_vector in zip(AGE_GRID, age_vectors):
    shifted_umap = project_embeddings(token_embeddings + age_vector, token_meta, random_state=UMAP_RANDOM_STATE)
    shifted_umap['age_years'] = age_value
    age_projection_frames.append(shifted_umap)

age_conditioned_umap = pd.concat(age_projection_frames, ignore_index=True)
